# Quantization Aware Training + Knowledge Distillation 
## PyTorch -> ONNX -> TfLite

In [2]:
import os
import torch

from src.utils import load_data, process_callbacks
from src.quantization.converters.to_onnx import export_pytorch_to_onnx, export_onnx_to_savedmodel, export_savedmodel_to_tflite
from src.train.knowledge_distillation import train_qat_kd
from src.utils.logging_setup import configure_logging

In [3]:
# Specify random seed for repeatable results
_ = torch.manual_seed(191009)

configure_logging(True)

### Hyperparameters

In [ ]:
dataset = "SkinCancer"
batch_size = 32
learning_rate = 0.001
epochs = 1
save_dir = f"models/{dataset}/Quantized"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher_model = "mobilenet_v2"
student_model = "mobilenet_v2"
args = {"callbacks": ["ModelCheckpoint", "EarlyStopping", "ReduceLROnPlateau"], 
        "save_dir": save_dir, 
        "model_name": student_model,
        "save_name": "qat_kd"}
CALLBACKS = list(process_callbacks(args).values())

os.makedirs(save_dir, exist_ok=True)

dataloaders = load_data(dataset=dataset, batch_size=batch_size)

teacher_model_weights = 'models/SkinCancer/mobilenet_v2_best_model.pth'  # Update with your saved model weights

criterion = "cross_entropy"
optimizer = "adam"

teacher, quantized_student = train_qat_kd(
                                        teacher_name=teacher_model, 
                                        student_name=student_model,
                                        data_loaders=dataloaders,
                                        save_dir=save_dir,
                                        learning_rate=learning_rate,
                                        epochs=epochs,
                                        criterion = criterion,
                                        optimizer = optimizer,
                                        callbacks=CALLBACKS,
                                        quant_mode="export",
                                        teacher_model_weights=teacher_model_weights,
                                        device=device,
                                )


[INFO 2025-04-23 18:15:35,212 preprocessing.py:17] Train dataset size: 29322
[INFO 2025-04-23 18:15:35,213 preprocessing.py:26] Class distribution for train dataset:
[INFO 2025-04-23 18:15:35,213 preprocessing.py:28]   Class 'Actinic keratoses': 693 samples
[INFO 2025-04-23 18:15:35,213 preprocessing.py:28]   Class 'Basal cell carcinoma': 2658 samples
[INFO 2025-04-23 18:15:35,214 preprocessing.py:28]   Class 'Benign keratosis-like lesions': 2099 samples
[INFO 2025-04-23 18:15:35,214 preprocessing.py:28]   Class 'Chickenpox': 900 samples
[INFO 2025-04-23 18:15:35,214 preprocessing.py:28]   Class 'Cowpox': 792 samples
[INFO 2025-04-23 18:15:35,214 preprocessing.py:28]   Class 'Dermatofibroma': 191 samples
[INFO 2025-04-23 18:15:35,214 preprocessing.py:28]   Class 'HFMD': 1932 samples
[INFO 2025-04-23 18:15:35,215 preprocessing.py:28]   Class 'Healthy': 1368 samples
[INFO 2025-04-23 18:15:35,215 preprocessing.py:28]   Class 'Measles': 660 samples
[INFO 2025-04-23 18:15:35,215 preprocessi

Model prepared using Export Mode QAT.


Train Epoch 1/1:  81%|████████  | 739/917 [02:29<00:36,  4.85it/s, loss=0.8791]

### 2. Export to ONNX

In [ ]:
example_inputs = next(iter(dataloaders['train']))[0]
onnx_path = os.path.join(save_dir, f"{student_model}_qat_kd.onnx")
export_pytorch_to_onnx(quantized_student, example_inputs, onnx_path)

### 3. Convert ONNX → TensorFlow SavedModel

In [ ]:
save_dir = f"models/{dataset}/Quantized"
student_model = "mobilenet_v2"
dataloaders = load_data(dataset=dataset, batch_size=batch_size)

In [ ]:
tf_saved_model_dir = os.path.join(save_dir, f"{student_model}_qat_kd")
onnx_path = os.path.join(save_dir, f"{student_model}_qat_kd.onnx")
export_onnx_to_savedmodel(onnx_path, tf_saved_model_dir)

### 4. Convert SavedModel → TFLite

In [ ]:
tflite_path = os.path.join(save_dir, f"{student_model}_qat_kd.tflite")
export_savedmodel_to_tflite(tf_saved_model_dir, tflite_path, dataloaders["train"])

### 5. Model sizes

In [6]:
print(os.path.getsize("models/SkinCancer/Quantized/quantized_state.pth") / 1e6)
print(os.path.getsize("models/SkinCancer/Quantized/mobilenet_v2_qat_kd.onnx") / 1e6)
print(os.path.getsize("models/SkinCancer/Quantized/mobilenet_v2_qat_kd.tflite") / 1e6)

9.292454
9.891259
2.730944


In [ ]:
# TODO: Get weights from QAT and then use post-training quantization
# TODO: Try a different model from mobilenetv2
# TODO: Try post-training quantization by using mobilenet large as a teacher and mobilenet small
# TODO: explore more optimization techniques (low-rank aproximation)
print(os.path.getsize("models/SkinCancer/Quantized/quantized_student_state.pth") / 1e6)